# Profiling

Profiling helps you measure and optimize the performance of your workflows. This tutorial covers how to profile latency, token usage, and costs.

## What You'll Learn

1. Why profiling matters
2. Configuring profilers
3. Running profiling via CLI
4. Analyzing profiling results
5. Common optimization strategies

## Key Metrics

| Metric | Description | Why It Matters |
|--------|-------------|----------------|
| **Latency** | Time to complete request | User experience |
| **Token Usage** | Input/output tokens | Cost |
| **Tool Calls** | Number of tool invocations | Efficiency |
| **LLM Calls** | Number of LLM invocations | Cost, latency |


In [ ]:
import sys
from pathlib import Path

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


## 1. Create a Workflow to Profile


In [ ]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.tool.datetime_tools import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

time_tool = CurrentTimeTool(name="current_time")

try:
    from nat_simple_calculator.register import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    tools = [time_tool, calculator]
except ImportError:
    tools = [time_tool]

agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
)

workflow = NatWorkflow(entrypoint=agent)
print("✅ Workflow created")


## 2. Configure Profiling

Profiling is configured as part of the evaluation configuration:


In [ ]:
import json

from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import EvalOutputConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation
from nat.utils.sdk.nat_evaluation import ProfilerConfig

# Create test dataset
test_data = [
    {"input": "What is 2 + 2?", "expected_output": "4"},
    {"input": "What is 10 * 5?", "expected_output": "50"},
    {"input": "What time is it?", "expected_output": ""},
]

data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)
dataset_path = data_dir / "profiling_dataset.json"
with open(dataset_path, "w") as f:
    json.dump(test_data, f, indent=2)

# Configure evaluation with profiler enabled
evaluation = NatEvaluation(
    output_dir=Path("./profiling_results"),
    dataset=EvalDatasetJsonConfig(file_path=dataset_path),
    output=EvalOutputConfig(
        include_io=True,       # Include input/output in results
        include_trace=True,    # Include execution trace
    ),
    profiler=ProfilerConfig(
        enabled=True,          # Enable profiling
    ),
)

workflow.add_evaluator(evaluation)
print("✅ Profiler configured")


## 3. Save Configuration


In [ ]:
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "profiling_workflow.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Saved to: {config_path}")


## 4. Run Profiling via CLI

```bash
# Run evaluation with profiling enabled
nat eval --config_file configs/profiling_workflow.yaml

# Run multiple repetitions for statistical significance
nat eval --config_file configs/profiling_workflow.yaml --reps 5
```


## 5. Understanding Profiling Results

After profiling, you'll find results in the output directory:

```
profiling_results/
├── profiling_summary.json    # Aggregated metrics
├── profiling_details.json    # Per-request details
└── eval_results.json         # Full evaluation results
```

### Example Profiling Summary

```json
{
  "summary": {
    "total_requests": 3,
    "avg_latency_ms": 2450,
    "p50_latency_ms": 2100,
    "p95_latency_ms": 3200,
    "total_tokens": 1500,
    "avg_tokens_per_request": 500,
    "total_llm_calls": 9,
    "avg_llm_calls_per_request": 3,
    "total_tool_calls": 6,
    "avg_tool_calls_per_request": 2
  }
}
```

### Per-Request Details

```json
{
  "requests": [
    {
      "input": "What is 2 + 2?",
      "latency_ms": 2100,
      "input_tokens": 150,
      "output_tokens": 80,
      "llm_calls": 2,
      "tool_calls": 1,
      "tools_used": ["calculator.add"]
    }
  ]
}
```


## Optimization Strategies

Based on profiling results, consider these optimizations:

### High Latency
- Use faster models (e.g., `llama-3.1-8b` instead of `70b`)
- Reduce `max_tokens` if responses are shorter than limit
- Use Tool Calling agent instead of ReAct for simple tasks

### High Token Usage
- Simplify system prompts
- Filter tool descriptions to only relevant ones
- Use `include` to limit available functions

### Too Many LLM Calls
- Improve prompts to get correct answers faster
- Use ReWOO agent for multi-step tasks (plans upfront)
- Add better error handling

## Summary

✅ **ProfilerConfig** - Enable profiling in evaluation  
✅ **Metrics** - Latency, tokens, LLM calls, tool calls  
✅ **CLI** - `nat eval` runs profiling  
✅ **Results** - Summary and per-request details  

## Next Steps

- **[11_optimization.ipynb](./11_optimization.ipynb)** - Automated optimization
- **[12_observability.ipynb](./12_observability.ipynb)** - Detailed tracing
